In [27]:
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_community.document_loaders import PyPDFLoader
from datasets import Dataset
from ragas import evaluate
from ragas.metrics.collections import faithfulness, answer_relevancy, context_recall, context_precision
from ragas import RunConfig
import os
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter
from dotenv import load_dotenv
from pdf2image import convert_from_path
from ragas.testset.graph import Node, KnowledgeGraph
import io
import base64
from ragas.testset.graph import Node
from ragas.testset.synthesizers import default_query_distribution
from ragas.testset.transforms import SummaryExtractor, EmbeddingExtractor, apply_transforms

In [3]:
load_dotenv()

True

In [4]:
rate_limiter = InMemoryRateLimiter(
    requests_per_second=7,
    check_every_n_seconds=0.1,
    max_bucket_size=10
)

In [5]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [6]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, rate_limiter=rate_limiter)

In [25]:
transforms = [
    SummaryExtractor(llm=LangchainLLMWrapper(llm)),
    EmbeddingExtractor(embedding_model=LangchainEmbeddingsWrapper(embeddings))
]

C:\Users\UserAdmin\AppData\Local\Temp\ipykernel_10664\160981847.py:2: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  SummaryExtractor(llm=LangchainLLMWrapper(llm)),
C:\Users\UserAdmin\AppData\Local\Temp\ipykernel_10664\160981847.py:3: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  EmbeddingExtractor(embedding_model=LangchainEmbeddingsWrapper(embeddings))


In [8]:
pages = convert_from_path("pdfs/government-data-security-policies.pdf", dpi=200)

In [11]:
def pil_to_b64(img):
    buffered = io.BytesIO()
    img.save(buffered, format="PNG")
    img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")
    return img_str

In [29]:
nodes = []
for i, page_img in enumerate(pages):
    b64_data = pil_to_b64(page_img)
    nodes.append(
        Node(
            type="chunk",
            properties={
                "page_content": "",
                "page_number": i + 1,
                "image_base64": b64_data,
                "modality": "image"
            }
        )
    )

kg = KnowledgeGraph(nodes=nodes)

In [32]:
apply_transforms(kg, transforms)

Applying SummaryExtractor:   0%|          | 0/22 [00:00<?, ?it/s]Task failed with IndexError: list index out of range
Task failed with IndexError: list index out of range
Task failed with IndexError: list index out of range
Task failed with IndexError: list index out of range
Task failed with IndexError: list index out of range
Task failed with IndexError: list index out of range
Task failed with IndexError: list index out of range
Task failed with IndexError: list index out of range
Task failed with IndexError: list index out of range
Task failed with IndexError: list index out of range
Task failed with IndexError: list index out of range
Task failed with IndexError: list index out of range
Task failed with IndexError: list index out of range
Task failed with IndexError: list index out of range
Task failed with IndexError: list index out of range
Task failed with IndexError: list index out of range
Task failed with IndexError: list index out of range
Task failed with IndexError: list 

IndexError: list index out of range

In [21]:
generator = TestsetGenerator(
    llm=LangchainLLMWrapper(llm),
    embedding_model=LangchainEmbeddingsWrapper(embeddings),
    knowledge_graph=kg
)

C:\Users\UserAdmin\AppData\Local\Temp\ipykernel_10664\1325986440.py:2: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  llm=LangchainLLMWrapper(llm),
C:\Users\UserAdmin\AppData\Local\Temp\ipykernel_10664\1325986440.py:3: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embedding_model=LangchainEmbeddingsWrapper(embeddings),


In [22]:
query_distribution = default_query_distribution(LangchainLLMWrapper(llm))

C:\Users\UserAdmin\AppData\Local\Temp\ipykernel_10664\2737032125.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  query_distribution = default_query_distribution(LangchainLLMWrapper(llm))


In [23]:
dataset = generator.generate(testset_size=10, query_distribution=query_distribution)

ValueError: No nodes that satisfied the given filer. Try changing the filter.

In [11]:
test_df.to_csv("ragas_testset.csv", index=False)

In [ ]:
test_df.head()

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,What is the intended audience for the Governme...,[Government Data Security Policies |  1\nGO...,The Government Data Security Policies document...,Information Security Analyst,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
1,Wht are the key policies in the Govenment's da...,[The Government takes its responsibility as a ...,The key policies in the Government's data secu...,Data Security Analyst,MISSPELLED,LONG,single_hop_specific_query_synthesizer
2,What steps should agencies take for effective ...,[Section 1: Data Security Risk Management\n01 ...,To ensure adequate and effective data security...,Information Security Analyst,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
3,How can agencies minimize the surface area of ...,[Section 3: Reduce the surface area of attack ...,Agencies can minimize the surface area of atta...,Information Security Analyst,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
4,How do agencies log and monitor data transacti...,[<1-hop>\n\nSection 4: Log and monitor data \n...,Agencies log and monitor data transactions to ...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
